In [15]:
## Data obtain through RDAS

import os
import csv 
from datetime import datetime 
import time
import mysql.connector

# # Start time
start_time = time.time()

import mysql.connector

class PeopleFecther:

    def __init__(self):
        pass

    def mysql_conn(self):

        try:
            return mysql.connector.connect(
                host='ncatslnmadbpdv04.ncats.nih.gov', 
                port='3306', 
                user='rdas_user',
                password='RDASuser123$',
                database='rdas_db',
                collation="utf8mb4_unicode_ci",  #Choose compatible collation
                charset="utf8mb4"# Add this to your connection string
            )
        except mysql.connector.Error as err:
            print(f'MySQL connection error: {err}')



    def fetch_people_by_last_name(self, conn, last_name):

        query = f'''SELECT associate_id, associate_type, source, first_name,last_name,
            affiliation, orc_id, email, phone,
            '' AS first_publication_date,
            '' AS title,
            '' AS abstract_text,
            '' AS author_list

            FROM rdas_db.person_of_all_sources where last_name=\'{last_name}\'
        '''
        result_list = []

        #1. Publication
        associate_type = 'Publication'
        query_publication = f'''  
            WITH publication_data AS (
                SELECT
                    ps.id, ps.associate_id, ps.associate_type, ps.source,
                    ps.first_name, ps.last_name, ps.affiliation,
                    ps.orc_id, ps.email, ps.phone,
                    pa.first_publication_date, pa.title, pa.abstract_text,
                    (
                        SELECT GROUP_CONCAT(sub.first_name, ' ', sub.last_name  SEPARATOR ', ')
                        FROM rdas_db.person_of_all_sources AS sub
                        WHERE
                            sub.source = '{associate_type}'
                            AND sub.associate_type = 'Author'
                            AND sub.associate_id = ps.associate_id
                    ) AS author_list

                FROM rdas_db.person_of_all_sources AS ps
                JOIN rdas_db.publication_article AS pa ON ps.associate_id = pa.pubmed_id
                WHERE
                    ps.last_name = '{last_name}'
                    AND ps.source = '{associate_type}'
            )
            SELECT
                *,
                TRIM(SUBSTRING_INDEX(author_list, ',', 1)) AS first_author,
                TRIM(SUBSTRING_INDEX(author_list, ',', -1)) AS last_author
            FROM publication_data
        '''

        results_1 = self.fetch(conn, query_publication, associate_type)
        result_list.extend(results_1)

        # 2. Clinical Trial
        associate_type = 'ClinicalTrial'
        #SELECT GROUP_CONCAT(DISTINCT sub.first_name, ' ', sub.last_name SEPARATOR ', ')
        query_clinical_trial = f'''  
           WITH clinical_trial_data AS (
                SELECT
                    ps.id, ps.associate_id, ps.associate_type, ps.source,
                    ps.first_name, ps.last_name, ps.affiliation,
                    ps.orc_id, ps.email, ps.phone,
                    '' AS first_publication_date, 
                    '' AS title, 
                    '' AS abstract_text,
                    (
                        SELECT GROUP_CONCAT(sub.first_name, ' ', sub.last_name SEPARATOR ', ')
                        FROM rdas_db.person_of_all_sources AS sub
                        WHERE
                            sub.source = '{associate_type}'
                            AND sub.associate_id = ps.associate_id
                    ) AS author_list
                FROM rdas_db.person_of_all_sources AS ps
                WHERE
                    ps.last_name = '{last_name}'
                    AND ps.source = '{associate_type}'
            )
            SELECT
                *,
                TRIM(SUBSTRING_INDEX(author_list, ',', 1)) AS first_author,
                TRIM(SUBSTRING_INDEX(author_list, ',', -1)) AS last_author
            FROM clinical_trial_data
        '''

        results_2 = self.fetch(conn, query_clinical_trial, associate_type)
        result_list.extend(results_2)

        # 3. Grant
        associate_type = 'GrantProject'
        query_grant = f'''  
            WITH grant_project_data AS (
                SELECT
                    ps.id, ps.associate_id, ps.associate_type, ps.source,
                    ps.first_name, ps.last_name, ps.affiliation,
                    ps.orc_id, ps.email, ps.phone,
                    '' AS first_publication_date, 
                    gp.project_title AS title, 
                    '' AS abstract_text,
                    (
                        SELECT GROUP_CONCAT(sub.first_name, ' ', sub.last_name SEPARATOR ', ')
                        FROM rdas_db.person_of_all_sources AS sub
                        WHERE
                            sub.source = '{associate_type}'
                            AND sub.associate_id = ps.associate_id
                    ) AS author_list
                FROM rdas_db.person_of_all_sources AS ps
                JOIN rdas_db.grant_project AS gp ON ps.associate_id = gp.application_id
                WHERE
                    ps.last_name = '{last_name}'
                    AND ps.source = '{associate_type}'
            )
            SELECT
                *,
                TRIM(SUBSTRING_INDEX(author_list, ',', 1)) AS first_author,
                TRIM(SUBSTRING_INDEX(author_list, ',', -1)) AS last_author
            FROM  grant_project_data
        '''
        results_3 = self.fetch(conn, query_grant, associate_type)
        result_list.extend(results_3)

        return result_list


    def fetch(self, conn, query, associate_type):
        
        cursor = conn.cursor(dictionary=True)
        cursor.execute(query) 

        result_list = []
        batch_num = 0
        batch_size = 100
        while True:

            batch = cursor.fetchmany(batch_size)
            if not batch:
                print('\n\n---------------- All data retrieved, no more data ----------------\n\n')
                break

            batch_num += 1
            print(f'Batch [{associate_type}]#: {batch_num} - ')
            result_list.extend(batch)

        cursor.close()

        return result_list



if __name__ == '__main__':
    
    # 1. init
    fetcher = PeopleFecther()

    conn = fetcher.mysql_conn()

    #2. Query
    last_name = 'Moon'
    results = fetcher.fetch_people_by_last_name(conn, last_name)

    if conn:
        conn.close()

    #3. Generate file
    headers = ['id','associate_id', 'associate_type', 'source', 
               'first_name', 'last_name', 'affiliation', 
               'orc_id', 'email', 'phone', 
               'first_publication_date', 'title', 'abstract_text', 'first_author', 'last_author', 'author_list']
    
    file_name = f'{last_name}-{datetime.today().strftime("%Y-%m-%d")}.csv'
    if os.path.exists(file_name):
        os.remove(file_name)

    with open(file_name, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=headers)

        # Write the header row
        writer.writeheader()

        # Write the data rows
        writer.writerows(results)


    print(f"CSV file '{file_name}' has been generated successfully.")

# End time
end_time = time.time()
elapsed = end_time - start_time
print(f"✔️ Done in {elapsed:.2f} seconds")

ModuleNotFoundError: No module named 'mysql'

In [2]:
## fix malformed CSV rows
import csv
import time

# Start time
start_time = time.time()

def fix_csv_with_csv_module(input_file, output_file, expected_fields=15):
    with open(input_file, 'r', encoding='utf-8-sig', newline='') as f_in, \
         open(output_file, 'w', encoding='utf-8-sig', newline='') as f_out:

        reader = csv.reader(f_in)
        writer = csv.writer(f_out)
             
        for row in reader:
            if len(row) > expected_fields:
                # Combine all fields from the 15th field to the end
                fixed_row = row[:expected_fields-1]  # first 14 fields
                combined_field = ','.join(row[expected_fields-1:])
                fixed_row.append(combined_field)
                writer.writerow(fixed_row)
            else:
                writer.writerow(row)

file_name = "Moon"
# Usage
fix_csv_with_csv_module(f"{file_name}-2025-09-22.csv", f"{file_name}_correction.csv")

In [3]:
# 1. Generate multiple versions of the combination (name_set consisted of full name as hyphen-removed)
# 2. Generated first initial and two initials
# 3. orc_group: Generated ORCID group based on same ORCID as a starting identity label. 
#               Within each ORCID group, removes rows whose initials do not match the majority, fixing inconsistent names tied to the same ORCID.
# 4. affil_group: Groups affil author group who share the same (first name & last name & affiliation) and appear at least twice, assigning them new group labels. 
# 5. coauthor_group: Groups authors who share the same (first name & last name & coauthor) and appear at least twice, assigning them new group labels.
# 6. Clears “group_1” (baseline ORCID group), then saves the processed dataset.

import pandas as pd
import ast
from collections import Counter

######################################################
# Load the dataset
#file_name = "Kim"
all_authors_file = file_name+"_correction.csv"
all_authors_initial_file = file_name+"_all_authors_initial.csv"

df = pd.read_csv(all_authors_file, encoding='utf-8-sig')

######################################################
# Parse all_authors column as list and SORT
df['author_list'] = df['author_list'].apply(
    lambda x: sorted([name.strip() for name in x.split(',')]) 
    if isinstance(x, str) and not pd.isna(x) 
    else (sorted(x) if isinstance(x, list) else [])
)

######################################################
# generated variations of first name 
def generate_name_set(first_name):
    if pd.isna(first_name) or not isinstance(first_name, str) or not first_name.strip():
        return set()
    
    name_set = set()
    
    # Clean up name
    clean_name = first_name.strip()
    name_set.add(clean_name)  # Full name
    name_set.add(clean_name.replace('-', ' '))
    
    # Split on hyphens, spaces
    parts = clean_name.replace('-', ' ').split()
    
    if parts:
        # Add initials like "S J"
        initials = ' '.join([p[0] for p in parts if p])
        if initials:
            name_set.add(initials)
        
        # Add single first initial like "S"
        name_set.add(parts[0][0])
    
    return sorted(name_set)

# Apply to your DataFrame
df['name_set'] = df['first_name'].apply(generate_name_set)

######################################################
# Assign group labels based on unique ORCID IDs
orc_id_to_group = {orc_id: f'group_{i+1}' for i, orc_id in enumerate(df['orc_id'].unique())}
df['orc_group'] = df['orc_id'].map(orc_id_to_group)

######################################################
## let's clean ORC if there are multiple initial - majority
max_group = df['orc_group'].nunique()

######################################################
## generated first_initial and two_initials
# Extract first initial from name_set
def get_first_initial(name_set):
    if not name_set:
        return None
    for name_variant in name_set:
        if name_variant and isinstance(name_variant, str):
            return name_variant[0].upper()
    return None

df['first_initial'] = df['name_set'].apply(get_first_initial)

for group, subdf in df.groupby('orc_group'):
    if group == "group_1": continue;
    initials = subdf['first_initial'].dropna().tolist()
    unique_initials = set(initials)

    if not initials:
        continue  # Skip empty groups

    # Only act on groups with multiple different initials
    if len(unique_initials) > 1:
        counter = Counter(initials)
        majority_initial, _ = counter.most_common(1)[0]
        
        # Identify rows NOT matching majority initial
        mask = (df['orc_group'] == group) & (df['first_initial'] != majority_initial)

        # Set their orc_group to empty string
        df.loc[mask, 'orc_group'] = ""

# Extract two initials from name_set
def get_two_initials(name_set):
    if not name_set:
        return None

    for name_variant in name_set:
        if name_variant and isinstance(name_variant, str):
            parts = name_variant.strip().split()
            if len(parts) >= 2:
                initials = ' '.join([p[0].upper() for p in parts if p])
                if len(initials.replace(' ', '')) == 2:
                    return initials
    return None

df['two_initials'] = df['name_set'].apply(get_two_initials)

######################################################
## groupby orc_group
for group, subdf in df.groupby('orc_group'):
    if group == "group_1":
        continue

    initials = subdf['two_initials'].dropna().tolist()
    unique_initials = set(initials)

    if not initials:
        continue  # Skip if no valid two-initials

    
    # Only act on groups with multiple different two-initials
    if len(unique_initials) > 1:
        counter = Counter(initials)
        majority_initial, _ = counter.most_common(1)[0]
        #print(group, counter.most_common(1)[0],counter.most_common(2)[0])

        # Identify rows NOT matching majority two_initials
        mask = (df['orc_group'] == group) & (df['two_initials'] != majority_initial)

        # Clear group assignment for inconsistent ones
        df.loc[mask, 'orc_group'] = ""
        
######################################################
# Assign group label based on unique same full name (include only one abbr) with affiliation
# Create a helper column for grouping (only if affiliation is not missing)
df['name_affil'] = list(zip(df['first_name'], df['last_name'], df['affiliation']))

# Filter out rows with missing or empty affiliation
valid_name_affil = df[~df['affiliation'].isna() & (df['affiliation'].str.strip() != '')]['name_affil']

# Count occurrences of valid combinations
name_affil_counts = valid_name_affil.value_counts()

# Keep only combinations with count >= 2
repeated_affils = [k for k, v in name_affil_counts.items() if v >= 2]

# Assign new group numbers starting from max_group + 1
name_affil_to_group = {
    name_affil: f'group_{i + max_group + 1}'
    for i, name_affil in enumerate(repeated_affils)
}

# Map only those combinations to the affil_group column
df['affil_group'] = df['name_affil'].map(name_affil_to_group)

# Drop helper column
df.drop(columns=['name_affil'], inplace=True)
######################################################
######################################################




######################################################
######################################################
# Assign group label based on unique same full name (include only one abbr) with co-authors (single coauthors, self, is excluded)
# Normalize coauthor list as a sorted tuple (only if more than one coauthor)
df['coauthor_set'] = df['author_list'].apply(
    lambda x: tuple(sorted(set(x))) if isinstance(x, list) and len(set(x)) > 1 else None
)

# Create a key based on first name, last name, and coauthor set (if valid)
df['name_coauthors_key'] = df.apply(
    lambda row: (row['first_name'], row['last_name'], row['coauthor_set']) if row['coauthor_set'] else None,
    axis=1
)

# Count how many times each combination appears
valid_keys = df['name_coauthors_key'].dropna()
key_counts = valid_keys.value_counts()

# Keep only combinations that occur more than once
repeated_keys = [k for k, v in key_counts.items() if v >= 2]

# Assign group numbers to repeated combinations
starting_group = max_group + len(name_affil_to_group)
coauthor_key_to_group = {
    key: f'group_{i + starting_group + 1}'
    for i, key in enumerate(repeated_keys)
}

# Map group number to new column
df['coauthor_group'] = df['name_coauthors_key'].map(coauthor_key_to_group)

# Clean up temporary columns
df.drop(columns=['coauthor_set', 'name_coauthors_key'], inplace=True)
######################################################
######################################################


df['orc_group'] = df['orc_group'].replace('group_1', '')

# Save the updated dataframe to a new CSV file
df.to_csv(all_authors_initial_file, index=False, encoding='utf-8-sig')


In [4]:
## This section detects cases where an ORCID group conflicts with identity groups formed from affiliation or coauthors.
## It finds affiliation groups and coauthor groups that contain more than one ORCID group removes their ORCID group
    
all_authors_initial_clean_file = file_name+"_all_authors_initial_clean.csv"

# Filter rows that have both orc_group and affil_group defined
valid_rows_affil = df[
    df['orc_group'].notna() & (df['orc_group'] != '') &
    df['affil_group'].notna() & (df['affil_group'] != '')
]

# Filter rows that have both orc_group and coauthor_group defined
valid_rows_coauth = df[
    df['orc_group'].notna() & (df['orc_group'] != '') &
    df['coauthor_group'].notna() & (df['coauthor_group'] != '')
]


# Group by affil_group and collect unique orc_group values per group
affil_to_orc_groups = valid_rows_affil.groupby('affil_group')['orc_group'].nunique()

# Group by coauthor_group and count unique orc_group values
coauth_to_orc_groups = valid_rows_coauth.groupby('coauthor_group')['orc_group'].nunique()


# Find affil_groups linked to more than one orc_group
shared_affils = affil_to_orc_groups[affil_to_orc_groups > 1].index.tolist()

# Identify coauthor_groups linked to more than one orc_group
shared_coauths = coauth_to_orc_groups[coauth_to_orc_groups > 1].index.tolist()


# Filter the original DataFrame for those affil_groups
conflict_df_affils = valid_rows_affil[valid_rows_affil['affil_group'].isin(shared_affils)]

# Filter rows that belong to these shared coauthor groups
conflict_df_coauth = valid_rows_coauth[valid_rows_coauth['coauthor_group'].isin(shared_coauths)]


print(conflict_df_affils[["orc_id", "orc_group", "affil_group", "coauthor_group"]])
print(conflict_df_coauth[["orc_id", "orc_group", "affil_group", "coauthor_group"]])


# Get the orc_groups involved in the conflicting rows in the conflict set
conflict_orc_groups = conflict_df_affils['orc_group'].unique()
conflict_affil_groups = set(shared_affils)

# Find orc_groups that are *only* found in these conflicting coauthor_groups
conflict_orc_groups = conflict_df_coauth['orc_group'].unique()
conflict_coauthor_groups = set(shared_coauths)


# Count how many times each orc_group appears in the full DataFrame
orc_group_counts = df['orc_group'].value_counts()
# Get orc_groups that occur only once in the entire DataFrame
unique_orc_groups = orc_group_counts[orc_group_counts == 1].index.tolist()


# Filter conflict_df_coauth to only include those unique orc_groups
unique_conflict_affil_rows = conflict_df_affils[conflict_df_affils['orc_group'].isin(unique_orc_groups)]
unique_conflict_coauthor_rows = conflict_df_coauth[conflict_df_coauth['orc_group'].isin(unique_orc_groups)]

# # Output
# print(f"Number of orc_groups that appear only once: {len(unique_conflict_affil_rows)}")
# print(unique_conflict_affil_rows[["orc_id", "orc_group", "affil_group", "coauthor_group"]])

# print(f"Number of orc_groups that appear only once: {len(unique_conflict_coauthor_rows)}")
# print(unique_conflict_coauthor_rows[["orc_id", "orc_group", "affil_group", "coauthor_group"]])

# Now filter the conflict_df for those unique orc_groups
unique_conflict_affil_rows = unique_conflict_affil_rows[unique_conflict_affil_rows['orc_group'].isin(unique_orc_groups)]
unique_conflict_coauthor_rows = unique_conflict_coauthor_rows[unique_conflict_coauthor_rows['orc_group'].isin(unique_orc_groups)]

# Step: Set orc_group to "" for matching rows in the original DataFrame
df.loc[df['orc_group'].isin(unique_conflict_affil_rows['orc_group']), 'orc_group'] = ''
df.loc[df['orc_group'].isin(unique_conflict_coauthor_rows['orc_group']), 'orc_group'] = ''
df.to_csv(all_authors_initial_clean_file, index=False, encoding='utf-8-sig')


                   orc_id  orc_group affil_group coauthor_group
5369  0000-0002-9819-0775  group_123   group_665            NaN
5533  0000-0003-2550-913X  group_134   group_432            NaN
5579  0000-0001-9745-679X  group_138   group_762     group_1293
5580  0000-0002-7338-0652  group_139   group_762     group_1293
5780  0000-0003-2550-913X  group_134   group_432            NaN
6781  0000-0002-6642-5699  group_227   group_940     group_1286
6782  0000-0003-1858-0261   group_19   group_940     group_1286
7684   0000-0003-2550-913  group_283   group_432     group_1063
8267  0000-0003-2550-913X  group_134   group_432            NaN
8272  0000-0003-2550-913X  group_134   group_432     group_1063
8438  0000-0003-1866-2910   group_53   group_665            NaN
                   orc_id  orc_group affil_group coauthor_group
5579  0000-0001-9745-679X  group_138   group_762     group_1293
5580  0000-0002-7338-0652  group_139   group_762     group_1293
6781  0000-0002-6642-5699  group_227   g

In [5]:
## relevant to "merge_group"
## This section creates a merge_group by combining the identity signals from orc_group, affil_group, and coauthor_group.
## It starts with ORCID groups, then assigns affiliation groups to the same merged identity when their names overlap strongly or when they consistently map to a single ORCID group. 
## It then applies the same logic to coauthor groups, linking them to an existing merged identity or creating a fallback group when ORCID data is missing. 

all_authors_initial_clean_merged_file = file_name+"_all_authors_initial_clean_merged.csv"

df['merge_group'] = df['orc_group'].fillna('')

# ====================
# Map affil_group to representative orc_group or fallback to affil_group
# ====================

# Rows with valid affil_group
valid_affil = df[(df['affil_group'].notna()) & (df['affil_group'] != '')]

# Determine affil_groups with any corresponding orc_group
affil_with_orc = valid_affil[valid_affil['orc_group'].notna() & (valid_affil['orc_group'] != '')]
#affil_to_orc = affil_with_orc.groupby('affil_group')['orc_group'].first().to_dict()
affil_to_orc = affil_with_orc.groupby('affil_group')['orc_group'].apply(list).to_dict()

#####################
# Group and get all orc_groups as a list
affil_to_orc = affil_with_orc.groupby('affil_group')['orc_group'].apply(list).to_dict()

# Filter to only those with multiple unique orc_groups
dub_affil_to_orc = {
    affil: list(set(orc_list))
    for affil, orc_list in affil_to_orc.items()
    if len(set(orc_list)) > 1
}

#print(affil_to_orc)
#####################

# Step: Refine affil_to_orc mapping using name_set overlap
refined_affil_to_orc = {}

# Group affil_group with missing orc_group but strong internal name overlap (more than three initial)
affil_groups_with_no_orc = df[(df['orc_group'].isna()) | (df['orc_group'] == '')]['affil_group'].dropna().unique()

for affil in affil_groups_with_no_orc:
    affil_rows = df[df['affil_group'] == affil]
    name_sets = affil_rows['name_set'].dropna().tolist()
    
    # Skip if there's not enough name data
    if len(name_sets) < 2:
        continue

    overlap = len(set.intersection(*map(set, name_sets)))
    
    if overlap >= 3:
        # Check for unique non-empty orc_group
        unique_orc_groups = affil_rows['orc_group'].dropna().unique()
        unique_orc_groups = [og for og in unique_orc_groups if og != '']
        
        if len(unique_orc_groups) == 1:
            merge_group_name = unique_orc_groups[0]
            df.loc[df['affil_group'] == affil, 'merge_group'] = merge_group_name
        elif len(unique_orc_groups) == 0:
            merge_group_name = f"{affil}"
        else: continue;
            #merge_group_name = f""
            #merge_group_name = f"pseudo_orc_{affil}"
            #print(len(unique_orc_groups))
        
        df.loc[df['affil_group'] == affil, 'merge_group'] = merge_group_name



# Get affil_groups that have NO associated orc_group anywhere in the DataFrame
affil_without_orc = set(valid_affil['affil_group'].unique()) - set(affil_to_orc.keys())

# Update only using refined mapping with overlap condition
affil_mask_with_refined_orc = df['affil_group'].isin(refined_affil_to_orc.keys())
#df.loc[affil_mask_with_refined_orc, 'merge_group'] = df.loc[affil_mask_with_refined_orc, 'affil_group'].map(refined_affil_to_orc)
df.loc[affil_mask_with_refined_orc, 'merge_group'] = df.loc[affil_mask_with_refined_orc, 'orc_group'].map(refined_affil_to_orc)


# For affil_groups with NO corresponding orc_group, set merge_group = affil_group
affil_mask_without_orc = df['affil_group'].isin(affil_without_orc)
df.loc[affil_mask_without_orc, 'merge_group'] = df.loc[affil_mask_without_orc, 'affil_group']

# ====================
# Map coauthor_group to representative merge_group or fallback to coauthor_group
# ====================

# Rows with valid coauthor_group
valid_coauth = df[(df['coauthor_group'].notna()) & (df['coauthor_group'] != '')]

# Determine representative merge_group per coauthor_group (if one already exists)
coauth_with_merge = valid_coauth[valid_coauth['merge_group'].notna() & (valid_coauth['merge_group'] != '')]
coauth_to_orc = coauth_with_merge.groupby('coauthor_group')['merge_group'].apply(list).to_dict()


# Step: Refine coauth_to_orc mapping using name_set overlap
refined_coauth_to_orc = {}

# Group coauth_group with missing orc_group but strong internal name overlap
coauth_groups_with_no_orc = df[(df['orc_group'].isna()) | (df['orc_group'] == '')]['coauthor_group'].dropna().unique()

for coauth in coauth_groups_with_no_orc:
    coauth_rows = df[df['coauthor_group'] == coauth]
    name_sets = coauth_rows['name_set'].dropna().tolist()
    
    # Skip if there's not enough name data
    if len(name_sets) < 2:
        continue

    overlap = len(set.intersection(*map(set, name_sets)))
    
    if overlap >= 3:
        # Check for unique non-empty orc_group
        unique_orc_groups = coauth_rows['orc_group'].dropna().unique()
        unique_orc_groups = [og for og in unique_orc_groups if og != '']
        
        if len(unique_orc_groups) == 1:
            merge_group_name = unique_orc_groups[0]
            df.loc[df['coauthor_group'] == coauth, 'merge_group'] = merge_group_name
        elif len(unique_orc_groups) == 0:
            merge_group_name = f"{coauth}"
        else: continue;
            #print (
            #merge_group_name = f""
            #merge_group_name = f"pseudo_orc_{coauth}"
            #print(len(unique_orc_groups))

        df.loc[df['coauthor_group'] == coauth, 'merge_group'] = merge_group_name


# Get coauthor_groups that have NO associated merge_group anywhere
coauth_without_orc = set(valid_coauth['coauthor_group'].unique()) - set(coauth_to_orc.keys())


# Update only using refined mapping with overlap condition
coauth_mask_with_refined_orc = df['coauthor_group'].isin(refined_coauth_to_orc.keys())
#df.loc[coauth_mask_with_refined_orc, 'merge_group'] = df.loc[coauth_mask_with_refined_orc, 'coauthor_group'].map(refined_coauth_to_orc)
df.loc[coauth_mask_with_refined_orc, 'merge_group'] = df.loc[coauth_mask_with_refined_orc, 'orc_group'].map(refined_coauth_to_orc)

# For coauthor_groups with NO merge_group, fallback to using coauthor_group name
coauth_mask_without_orc = df['coauthor_group'].isin(coauth_without_orc)
df.loc[coauth_mask_without_orc, 'merge_group'] = df.loc[coauth_mask_without_orc, 'coauthor_group']
# print(len(coauth_mask_with_refined_orc[coauth_mask_with_refined_orc].index))
# print(len(coauth_mask_without_orc[coauth_mask_without_orc].index))


df.to_csv(all_authors_initial_clean_merged_file, index=False, encoding='utf-8-sig')

In [6]:
## relevant to "set_coauthor_group"
## Strengthens identity merging by analyzing coauthor patterns after the merge phase.
## It first look each row’s coauthor list, then computes for every "merge_group" the authors who appear frequently within that group. 
## Groups that share enough high-frequency coauthors, share enough name elements (>=3), and share an author whose last name matches the target file name are considered the same person. 
## These groups are linked in a graph, merged into connected components, and each component is assigned a single unified identity. 
## Finally, the dataframe updates set_coauthor_group with these refined merged identifiers and saves the result.

import pandas as pd
import itertools
from itertools import combinations
from collections import Counter
from collections import defaultdict

all_authors_initial_clean_merged_coauthor_file = file_name+"_all_authors_initial_clean_merged_coauthor.csv"

# ---- Step 1: Preprocess sets ----
#df['all_authors'] = df['all_authors'].apply(lambda x: set(x) if isinstance(x, (list, set)) else set()) ##another fix?
#df['author_list'] = df['author_list'].apply(lambda x: sorted(ast.literal_eval(x)) if isinstance(x, str) else sorted(x)) ## fix!!
df['author_list'] = df['author_list'].apply(
    lambda x: sorted([name.strip() for name in x.split(',')]) 
    if isinstance(x, str) and not pd.isna(x) 
    else (sorted(x) if isinstance(x, list) else [])
)


# ---- Step 2: Initialize necessary columns ----
# Initialize coauthor group column
if 'set_coauthor_group' not in df.columns:
    df['set_coauthor_group'] = ''
else:
    df['set_coauthor_group'] = df['set_coauthor_group'].fillna('')


df['set_coauthor_group'] = df['merge_group'].fillna('')

# Initialize shared_set column to track overlapping authors
# if 'shared_set' not in df.columns:
#     df['shared_set'] = [set() for _ in range(len(df))]
# else:
#df['shared_set'] = df['shared_set'].apply(lambda x: set(x) if isinstance(x, (list, set)) else "")

#######

# Step 1: Calculate high-frequency authors per group and store
high_freq_by_group = {}

grouped = df.groupby('merge_group')

for group_name, group_df in grouped:
    if group_name == "":
        continue

    author_counter = Counter()
    for authors in group_df['author_list']:
        author_counter.update(authors)

    name_set = set().union(*[set(names) for names in group_df['name_set']])
    
    group_size = len(group_df)
    threshold = group_size / 2

    high_freq_authors = [author for author, count in author_counter.items() if count >= threshold]

    high_freq_by_group[group_name] = {
        "size": group_size,
        "high_freq_authors": set(high_freq_authors),  # Store as set for quick intersection
        "name_set": set(name_set),
        "author_counts": dict(author_counter)
    }


# Step 2: Find group pairs sharing at least 3 high-frequency authors
group_high_freq_sets = {g: info["high_freq_authors"] for g, info in high_freq_by_group.items()}

shared_high_freq_matches = []

# Loop over all combinations of group pairs
for (group1, info1), (group2, info2) in combinations(high_freq_by_group.items(), 2):
    authors1 = info1["high_freq_authors"]
    authors2 = info2["high_freq_authors"]
    
    shared_authors = authors1.intersection(authors2)
    
    # Condition 1: at least 3 shared authors - but so many same initals let's increase one
    cond1 = len(shared_authors) >= 4

    # Condition 2: any author matches 'Lastname xxx'
    cond2 = any(author.startswith(file_name + " ") for author in shared_authors)

    # Condition 3: at least 3 shared names
    name_set1 = info1["name_set"]
    name_set2 = info2["name_set"]
    shared_names = name_set1.intersection(name_set2)
    cond3 = len(shared_names) >= 3 ###### this part is the considered as number of shared coauthors!!!!!!

    if cond1 and cond2 and cond3:
        shared_high_freq_matches.append((group1, group2, shared_authors, shared_names))

for g1, g2, shared_authors, shared_names in shared_high_freq_matches:
    print(f"{g1} & {g2} share {len(shared_authors)} authors: {sorted(shared_authors)}")
    print(f"Shared names: {sorted(shared_names)}\n")
print("\n\n")


# Build an undirected graph from your shared_high_freq_matches
graph = defaultdict(set)

for g1, g2, _, _ in shared_high_freq_matches:
    graph[g1].add(g2)
    graph[g2].add(g1)

# DFS to find connected components
def dfs(node, visited, component):
    visited.add(node)
    component.add(node)
    for neighbor in graph[node]:
        if neighbor not in visited:
            dfs(neighbor, visited, component)

# Build components
visited = set()
connected_components = []

for node in graph:
    if node not in visited:
        component = set()
        dfs(node, visited, component)
        connected_components.append(component)

# Print result
for i, component in enumerate(connected_components, 1):
    print(f"Component {i}: {sorted(component)}")

# Build a mapping from each group to its smallest group in its component
group_to_smallest = {}

for component in connected_components:
    smallest = min(component)
    for group in component:
        group_to_smallest[group] = smallest

# Apply this mapping to update df['set_coauthor_group']
df['set_coauthor_group'] = df['merge_group'].apply(lambda g: group_to_smallest.get(g, g))


# Save to CSV
df.to_csv(all_authors_initial_clean_merged_coauthor_file, index=False, encoding='utf-8-sig')

In [7]:
## relevant to "unified_group"
## It first computes, for every existing group, the authors who frequently co-occur within that group. 
## Then for each unassigned row, it checks which group shares the strongest overlap in high-frequency coauthors and shared names. 
## If a group matches strongly enough, the row is assigned to that group, producing a  "unified_group".

import pandas as pd
import itertools
from collections import Counter
import itertools

all_authors_initial_clean_merged_coauthor_merged_file = file_name+"_all_authors_initial_clean_merged_coauthor_merged.csv"

df['unified_group'] = df['set_coauthor_group'].fillna('')

# Step 1: Calculate high-frequency authors per group and store
high_freq_by_group = {}

grouped = df.groupby('set_coauthor_group')

for group_name, group_df in grouped:
    if group_name == "":
        continue

    author_counter = Counter()
    for authors in group_df['author_list']:
        author_counter.update(authors)

    name_set = set().union(*[set(names) for names in group_df['name_set']])
    
    group_size = len(group_df)
    threshold = group_size / 2

    high_freq_authors = [author for author, count in author_counter.items() if count >= threshold]

    high_freq_by_group[group_name] = {
        "size": group_size,
        "high_freq_authors": set(high_freq_authors),  # Store as set for quick intersection
        "name_set": set(name_set),
        "author_counts": dict(author_counter)
    }
    #print(f"{group_name} (size={group_size}): high-frequency authors: {high_freq_authors}")

def find_best_group(row_authors, row_names, high_freq_by_group):
    best_group = None
    best_overlap_count = 0

    if len(row_authors) > 100:
        return None

    for group_name, group_info in high_freq_by_group.items():
        author_overlap = row_authors.intersection(group_info['high_freq_authors'])
        name_overlap = row_authors.intersection(group_info['name_set'])

        # Condition: at least 3 shared high-frequency authors AND at least 3 shared names
        if len(author_overlap) >= 3 and len(name_overlap) >= 2:
            if len(author_overlap) > best_overlap_count:
                best_group = group_name
                best_overlap_count = len(author_overlap)
            elif len(author_overlap) == best_overlap_count and best_group is not None:
                best_group = min(best_group, group_name)  # Tie-break alphabetically

    return best_group
    

df['author_list'] = df['author_list'].apply(lambda x: set(x) if isinstance(x, (list, set)) else set())
df['name_set'] = df['name_set'].apply(lambda x: set(x) if isinstance(x, (list, set)) else set())

mask = df['set_coauthor_group'].isna() | (df['set_coauthor_group'] == '')

# Apply the function row-wise
df.loc[mask, 'unified_group'] = df[mask].apply(
    lambda row: find_best_group(row['author_list'], row['name_set'], high_freq_by_group),
    axis=1
)

# Save to CSV
df.to_csv(all_authors_initial_clean_merged_coauthor_merged_file, index=False, encoding='utf-8-sig')

In [8]:
## relevant to "firstlastauthor"
## To identifies recurring first–last author pairs with consistent name sets:
## Groups by (first_author & last_author & name_set) and counts occurrences.


import pandas as pd


all_authors_initial_clean_merged_coauthor_merged_firstlastauthor_file = file_name+"_all_authors_initial_clean_merged_coauthor_merged_firstlastauthor.csv"

# Load CSV
df = pd.read_csv(all_authors_initial_clean_merged_coauthor_merged_file)

# Drop missing values
df_filtered = df.dropna(subset=['first_author', 'last_author', 'name_set']).copy()

# Exclude self-pairings (firstAuthor == lastAuthor)
df_filtered = df_filtered[df_filtered['first_author'] != df_filtered['last_author']]

# # Convert unhashable 'set' in name_set to sorted tuples (which are hashable)
# df_filtered['name_set'] = df_filtered['name_set'].apply(
#     lambda x: tuple(sorted(x)) if isinstance(x, set) else x
# )

# Group by firstAuthor, lastAuthor, name_set
group_keys = ['first_author', 'last_author', 'name_set']
group_sizes = df_filtered.groupby(group_keys).size().reset_index(name='count')

# Only keep groups with more than 2 entries
valid_groups = group_sizes[group_sizes['count'] > 2].copy()

# Assign group names
valid_groups['firstlastauthor'] = [
    f'firstlastauthor_group_{i}' for i in range(len(valid_groups))
]

# Merge group labels back to filtered data
df_filtered = df_filtered.merge(valid_groups[group_keys + ['firstlastauthor']], on=group_keys, how='left')

# Merge into the original DataFrame
df = df.merge(
    df_filtered[['first_author', 'last_author', 'name_set', 'firstlastauthor']],
    on=['first_author', 'last_author', 'name_set'],
    how='left'
)

# Fill NaNs if needed
df['firstlastauthor'] = df['firstlastauthor'].fillna("")
df = df.drop_duplicates()

# Save result
df.to_csv(all_authors_initial_clean_merged_coauthor_merged_firstlastauthor_file, index=False)


In [9]:
## relevant to "unified_group_all"
## This step merges the unified coauthor groups with the first–last author groups 
## Groups by firstlastauthor and collects all unique unified_groups associated with each first–last author pair.
## Keeps only firstlastauthor groups that correspond to exactly one unique unified_group.
## Updates df['unified_group_all'] using this mapping, leaving other rows unchanged.


all_authors_initial_clean_merged_coauthor_merged_firstlastauthor_merged_file = file_name+"_all_authors_initial_clean_merged_coauthor_merged_firstlastauthor_merged.csv"


df['unified_group_all'] = df['unified_group'].fillna("")

# Filter out rows where 'unified_group' is empty or null, and where 'firstlastauthor' is empty or null
df_filtered = df[
    df['unified_group'].notna() & (df['unified_group'].str.strip() != '') &
    df['firstlastauthor'].notna() & (df['firstlastauthor'].str.strip() != '')
]

# Group by 'firstlastauthor' and aggregate unique 'unified_group's
grouped = df_filtered.groupby('firstlastauthor')['unified_group'].unique().reset_index()

unique_unified_groups = grouped[grouped['unified_group'].apply(len) == 1]
# Filter groups where count of unique 'unified_group' > 1

# Create a mapping from firstlastauthor to list of unified_groups
author_to_groups = dict(zip(unique_unified_groups['firstlastauthor'], unique_unified_groups['unified_group']))

# Update df['unified_group_all'] for rows where 'firstlastauthor' is in the mapping
def update_unified_group_all(row):
    if row['firstlastauthor'] in author_to_groups:
        # Convert array to comma-separated string or keep as list/set as needed
        return ','.join(sorted(author_to_groups[row['firstlastauthor']]))
    else:
        return row['unified_group_all']  # Keep original if no update

df['unified_group_all'] = df.apply(update_unified_group_all, axis=1)
print(unique_unified_groups, '\n\n')

            
# Save result
df = df.drop_duplicates()
df.to_csv(all_authors_initial_clean_merged_coauthor_merged_firstlastauthor_merged_file, index=False)

    


               firstlastauthor unified_group
0      firstlastauthor_group_1    [group_24]
3    firstlastauthor_group_102    [group_46]
4    firstlastauthor_group_103  [group_1039]
5    firstlastauthor_group_104    [group_24]
6    firstlastauthor_group_106  [group_1044]
..                         ...           ...
247   firstlastauthor_group_91   [group_154]
248   firstlastauthor_group_93    [group_53]
249   firstlastauthor_group_94    [group_46]
251   firstlastauthor_group_97  [group_1089]
252   firstlastauthor_group_99  [group_1149]

[183 rows x 2 columns] 




In [10]:
## relevant to "unified_group_all_other" and explain set of "new_common"
## This step handles remaining authors not yet assigned to a unified group, by finding groups with similar coauthors (“missed coauthor groups”).
## It groups rows by first name and checks for shared coauthors. 
## If 3 or more authors overlap in at least 3 rows, it assigns a new “missing” group (unified_group_all_other) and stores the shared coauthors. 

import pandas as pd
import ast

all_authors_initial_clean_merged_coauthor_merged_firstlastauthor_merged_misscoauthor_file = file_name+"_all_authors_initial_clean_merged_coauthor_merged_firstlastauthor_merged_misscoauthor.csv"

# Step 1: Filter the DataFrame
df_filtered = df[df["unified_group_all"] == ""].copy()

# Initialize new columns in the original df
df["unified_group_all_other"] = None
df["new_common"] = None

group_counter = 0

# Step 2: Group by first_name
for fname, group in df_filtered.groupby("first_name"):
    group = group.copy()
    n = len(group)
    assigned = set()
    rows = group.index.tolist()

    # Step 3: Iterate over row combinations
    for i in range(n):
        if rows[i] in assigned:
            continue
        group_members = [rows[i]]
        common_authors = ast.literal_eval(df.loc[rows[i], "author_list"])

        for j in range(i + 1, n):
            if rows[j] in assigned:
                continue
            compare_authors = ast.literal_eval(df.loc[rows[j], "author_list"])
            shared = common_authors & compare_authors

            if len(shared) >= 3:
                group_members.append(rows[j])
                common_authors &= shared

        # Step 4: Assign group if > 2 rows
        if len(group_members) > 2:
            group_counter += 1
            for idx in group_members:
                df.at[idx, "unified_group_all_other"] = f'missing_group_{group_counter}'
                df.at[idx, "new_common"] = list(common_authors)
                assigned.add(idx)

df.to_csv(all_authors_initial_clean_merged_coauthor_merged_firstlastauthor_merged_misscoauthor_file, index=False)

In [11]:
## relevant to "unified_group_new"
## This step merges multiple group assignments (unified_group_all and unified_group_other) into a single unified group column (unified_group_new) based on overlapping authors and names, handling both linked and unlinked groups.
## Build author/name dictionaries for existing groups (unified_group_all and unified_group_all_other).
## If they share at least some authors (≥ 3) and names, link them as related groups.
## Build clusters of linked groups using a graph approach (connected components).

all_authors_initial_clean_merged_coauthor_merged_firstlastauthor_merged_misscoauthor_merged_file = file_name+"_all_authors_initial_clean_merged_coauthor_merged_firstlastauthor_merged_misscoauthor_merged.csv"

import ast
import pandas as pd
from collections import Counter
from collections import defaultdict

def get_frequent_authors(df, group_col, group_name, author_col="author_list", min_count=1):
     # Filter group
    sub_df = df[df[group_col] == group_name].copy()

    # Parse string sets to real sets
    sub_df[author_col] = sub_df[author_col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("{") else x
    )

    # Flatten author lists
    all_authors = [
        author
        for author_list in sub_df[author_col]
        if isinstance(author_list, (set, list))
        for author in author_list
    ]

    # Count frequencies
    author_counts = Counter(all_authors)

    # Create result DataFrame
    result_df = pd.DataFrame(author_counts.items(), columns=["Author", "Count"])
    result_df = result_df[result_df["Count"] >= min_count]
    result_df = result_df.sort_values(by="Count", ascending=False).reset_index(drop=True)

    return result_df

def get_unique_names(df, group_col, group_name, names_col="name_set"):
    # Filter group
    sub_df = df[df[group_col] == group_name].copy()

    # Parse string-representations of sets/lists if needed
    sub_df[names_col] = sub_df[names_col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("{") else x
    )

    # Flatten and collect unique names
    unique_names = set(
        name
        for names_list in sub_df[names_col]
        if isinstance(names_list, (set, list))
        for name in names_list
    )

    return unique_names

unique_original_groups = df["unified_group_all"].dropna().unique()
unique_other_groups = df["unified_group_all_other"].dropna().unique()

# Prepare to store results
main_authors_dict = {}
main_names_dict = {}
other_authors_dict = {}
other_names_dict = {}

for main_group in unique_original_groups:
    main_authors_dict[main_group] = get_frequent_authors(df, "unified_group_all", main_group)
    main_names_dict[main_group] = get_unique_names(df, "unified_group_all", main_group)

for other_group in unique_other_groups:
    other_authors_dict[other_group] = get_frequent_authors(df, "unified_group_all_other", other_group)
    other_names_dict[other_group] = get_unique_names(df, "unified_group_all_other", other_group)
    

# Mapping group pairs into unified groups
group_links = []  # store tuples of (main_group, other_group)
for mg in unique_original_groups:
    if 'connected_group' not in mg: continue
    for og in unique_other_groups:
        overlap_name = main_names_dict[mg].intersection(other_names_dict[og])
        main_authors_filtered = set(main_authors_dict[mg][main_authors_dict[mg]['Count'] >= 2]['Author']) # = need condition
        other_authors_filtered = set(other_authors_dict[og][other_authors_dict[og]['Count'] >= 2]['Author']) # = need condition
        overlap_authors = main_authors_filtered.intersection(other_authors_filtered)

        if overlap_name and len(overlap_authors) > 3:
            group_links.append((mg, og))

# Step 1: Create connected components of groups
from collections import deque

def build_group_clusters(group_links):
    graph = defaultdict(set)
    for g1, g2 in group_links:
        graph[g1].add(g2)
        graph[g2].add(g1)

    visited = set()
    clusters = []
    for node in graph:
        if node not in visited:
            cluster = set()
            queue = deque([node])
            while queue:
                curr = queue.popleft()
                if curr in visited:
                    continue
                visited.add(curr)
                cluster.add(curr)
                queue.extend(graph[curr] - visited)
            clusters.append(cluster)
    return clusters

clusters = build_group_clusters(group_links)
print(clusters)


# Step 1: Assign cluster labels (same as before)
group_to_unified = {}
for idx, cluster in enumerate(clusters, start=1):
    unified_label = f"unified_group_{idx}"
    for group in cluster:
        group_to_unified[group] = unified_label

# Step 2: Handle unclustered groups (from both columns)
# Get all group names used
all_groups = pd.Series(pd.concat([df["unified_group_all"], df["unified_group_all_other"]])).dropna().unique()

# Track the next available index
next_index = len(set(group_to_unified.values())) + 1
#next_index = len(set(group_to_unified.values()))

# Assign new labels to unclustered groups
for group in all_groups:
    if group not in group_to_unified and group != "":
        group_to_unified[group] = file_name+f"_{next_index}"
        next_index += 1

# Step 3: Use .apply(assign_unified_group) as before
def assign_unified_group(row):
    group1 = row.get("unified_group_all")
    group2 = row.get("unified_group_all_other")
    if group1 in group_to_unified:
        return group_to_unified[group1]
    elif group2 in group_to_unified:
        return group_to_unified[group2]
    else:
        return None

df["unified_group_new"] = df.apply(assign_unified_group, axis=1)
df.loc[df['unified_group_new'] == 'unified_group_1', 'unified_group_new'] = file_name + '_0'

df["author_list"] = df["author_list"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("{") else x)
df["author_list"] = df["author_list"].apply(lambda x: len(x) if isinstance(x, (list, set)) and len(x) > 100 else x)
# df["shared_set"] = df["shared_set"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else x)
# df["shared_set"] = df["shared_set"].apply(lambda x: "xx" if isinstance(x, (list, set)) and len(x) > 30 else x)
df["new_common"] = df["new_common"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else x)
df["new_common"] = df["new_common"].apply(lambda x: len(x) if isinstance(x, (list, set)) and len(x) > 100 else x)
    
# Save to CSV
df.to_csv(all_authors_initial_clean_merged_coauthor_merged_firstlastauthor_merged_misscoauthor_merged_file, index=False)



[]


In [12]:

import csv
import os
import ast
import pandas as pd
from collections import Counter, defaultdict


# Function to get frequent authors
def get_frequent_authors(df, group_col, group_name, author_col="author_list", min_count=1):
    sub_df = df[df[group_col] == group_name].copy()
    sub_df[author_col] = sub_df[author_col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("{") else x
    )
    all_authors = [
        author.lower().replace("-", " ")
        for author_list in sub_df[author_col]
        if isinstance(author_list, (set, list))
        for author in author_list
    ]
    author_counts = Counter(all_authors)
    result_df = pd.DataFrame(author_counts.items(), columns=["Author", "Count"])
    result_df = result_df[result_df["Count"] >= min_count]
    result_df = result_df.sort_values(by="Count", ascending=False).reset_index(drop=True)
    return result_df

# Get unique names
def get_unique_names(df, group_col, group_name, names_col="name_set"):
    sub_df = df[df[group_col] == group_name].copy()
    sub_df[names_col] = sub_df[names_col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("{") else x
    )
    unique_names = set(
        name
        for names_list in sub_df[names_col]
        if isinstance(names_list, (set, list))
        for name in names_list
    )
    return unique_names

df["final"] = df["unified_group_new"]
unique_groups = df["unified_group_new"].dropna().unique()


# Build mapping of group -> frequent authors
group_to_authors = {}
target_groups = unique_groups
for group in target_groups:
    freq_authors_df = get_frequent_authors(df, "unified_group_new", group, "author_list", min_count=2)
    group_to_authors[group] = set(freq_authors_df["Author"])

# Build mapping of group -> unique names
group_to_names = {}
for group in target_groups:
    group_to_names[group] = get_unique_names(df, "unified_group_new", group)

# Containment analysis with BOTH conditions
from collections import defaultdict

containment_counts = defaultdict(list)

for group_a, authors_a in group_to_authors.items():
    names_a = group_to_names.get(group_a, set())

    for group_b, authors_b in group_to_authors.items():
        if group_a == group_b:
            continue

        names_b = group_to_names.get(group_b, set())
        name_overlap = names_a.intersection(names_b)

        if (
            authors_b and authors_b.issubset(authors_a)
            and len(name_overlap) >= 2
            and len(authors_b) >= 2
        ):
            containment_counts[group_a].append(group_b)

# Report most inclusive group(s)
#max_contained = max((len(v) for v in containment_counts.values()), default=0)

print("\n🏆 Most Inclusive Group(s) (with name overlap >= 3, author overlap >= 2):")
for group, contained_groups in containment_counts.items():
    #if len(contained_groups) == max_contained:
    if len(contained_groups) > 0:
        print(f"\n🔸 {group} contains {len(contained_groups)} groups:")
        for g in contained_groups:
            print(f"   - {g}")

# Build reverse mapping: contained_group -> container_group
contained_to_container = {}
for container, contained_list in containment_counts.items():
    for contained in contained_list:
        contained_to_container[contained] = container

# Update df["final"] accordingly
df["final"] = df["final"].apply(lambda x: contained_to_container.get(x, x))
df["final_dup"] = df["final"]


dfs = df.copy()
dfs.to_csv(file_name+"_final.csv", index=False)

# End time
end_time = time.time()
elapsed = end_time - start_time
print(f"✔️ Done in {elapsed:.2f} seconds")



🏆 Most Inclusive Group(s) (with name overlap >= 3, author overlap >= 2):

🔸 Moon_7 contains 2 groups:
   - Moon_6
   - Moon_1201

🔸 Moon_886 contains 1 groups:
   - Moon_156

🔸 Moon_893 contains 1 groups:
   - Moon_711

🔸 Moon_26 contains 3 groups:
   - Moon_326
   - Moon_370
   - Moon_440

🔸 Moon_29 contains 1 groups:
   - Moon_1232

🔸 Moon_33 contains 2 groups:
   - Moon_99
   - Moon_119

🔸 Moon_37 contains 2 groups:
   - Moon_1198
   - Moon_798

🔸 Moon_40 contains 1 groups:
   - Moon_289

🔸 Moon_901 contains 2 groups:
   - Moon_256
   - Moon_353

🔸 Moon_904 contains 1 groups:
   - Moon_51

🔸 Moon_912 contains 1 groups:
   - Moon_1283

🔸 Moon_64 contains 2 groups:
   - Moon_53
   - Moon_914

🔸 Moon_66 contains 1 groups:
   - Moon_53

🔸 Moon_914 contains 1 groups:
   - Moon_53

🔸 Moon_72 contains 1 groups:
   - Moon_275

🔸 Moon_920 contains 5 groups:
   - Moon_73
   - Moon_79
   - Moon_80
   - Moon_84
   - Moon_85

🔸 Moon_75 contains 1 groups:
   - Moon_73

🔸 Moon_78 contains 1 group

In [13]:
###### Moon : Done in 17.79 seconds
###### Ye   : Done in 125.46 seconds
###### Park : Done in 606.91 seconds
###### Zhu  : Done in 779.62 seconds
###### Zhao : Done in 1391.56 seconds
###### Kim  : Done in 4342.01 seconds